# Eksperimen Output Embedding

Eksperimen ini mengukur ukuran model embedding dan membandingkan output
embedding dari teks pendek dan panjang.

In [11]:
from __future__ import annotations

from pathlib import Path
from typing import Any

import pandas as pd
from fastembed import TextEmbedding
from IPython.display import display
from opensearchpy import OpenSearch
from tokenizers import Tokenizer

## 1. Konfigurasi

In [12]:
MODEL_NAME = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

LOREM_IPSUM_PARAGRAPH = (
    "Lorem ipsum dolor sit amet, consectetur adipiscing elit. Sed do eiusmod "
    "tempor incididunt ut labore et dolore magna aliqua. Ut enim ad minim "
    "veniam, quis nostrud exercitation ullamco laboris nisi ut aliquip ex ea "
    "commodo consequat. Duis aute irure dolor in reprehenderit in voluptate "
    "velit esse cillum dolore eu fugiat nulla pariatur. Excepteur sint occaecat "
    "cupidatat non proident, sunt in culpa qui officia deserunt mollit anim id "
    "est laborum."
)

SAMPLE_TEXTS = {
    "short": "Lorem ipsum dolor sit amet.",
    "long": " ".join([LOREM_IPSUM_PARAGRAPH] * 5),
}

## 2. Load model FastEmbed

Pada eksekusi pertama, FastEmbed akan mengunduh model ONNX ke cache lokal.
Eksekusi berikutnya menggunakan model dari cache tersebut.

In [13]:
model_catalog = next(
    item
    for item in TextEmbedding.list_supported_models()
    if item["model"].lower() == MODEL_NAME.lower()
)

print(f"Loading FastEmbed model {MODEL_NAME!r} ...")
model = TextEmbedding(model_name=MODEL_NAME)
model.embedding_size, model_catalog["dim"]

# Salin tokenizer tanpa truncation agar jumlah token asli tetap bisa dihitung.
token_limit = model.model.tokenizer.truncation["max_length"]
counting_tokenizer = Tokenizer.from_str(model.model.tokenizer.to_str())
counting_tokenizer.no_truncation()

Loading FastEmbed model 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2' ...


## 3. Hitung ukuran model lokal

FastEmbed belum menyediakan properti publik untuk direktori model spesifik.
`_model_dir` dipakai agar ukuran dihitung dari file yang benar-benar digunakan
untuk inference, bukan dari seluruh cache yang mungkin berisi model lain.

In [14]:
model_directory = Path(model.model._model_dir)
model_files = [path for path in model_directory.rglob("*") if path.is_file()]
model_size_bytes = sum(path.stat().st_size for path in model_files)

print(f"Model directory: {model_directory}")
print(f"Actual local model size: {model_size_bytes:,} bytes")

Model directory: C:\Users\Andakara\AppData\Local\Temp\fastembed_cache\models--qdrant--paraphrase-multilingual-MiniLM-L12-v2-onnx-Q\snapshots\faf4aa4225822f3bc6376869cb1164e8e3feedd0
Actual local model size: 252,138,706 bytes


## 4. Generate embedding teks pendek dan panjang

Hasil FastEmbed berupa NumPy array. Properti `dtype`, `shape`, dan `nbytes`
memberikan ukuran output numerik aktual tanpa mengasumsikan dimensinya.

In [15]:
sample_labels = list(SAMPLE_TEXTS)
sample_texts = list(SAMPLE_TEXTS.values())
embeddings = list(model.embed(sample_texts))

## 5. Hasil pengukuran

- `embedding_raw_bytes` adalah ukuran raw embedding yang relevan untuk estimasi
dasar penyimpanan vector.
- `token_limit_usage_pct` di atas 100% berarti teks dipotong saat embedding.

Ukuran raw embedding dihitung dari jumlah dimensi dikali ukuran setiap nilai:

```python
embedding_raw_bytes = embedding.size * embedding.itemsize
```

Pada NumPy, hasil perhitungan tersebut sudah tersedia melalui
`embedding.nbytes`. Contoh untuk output FastEmbed pada eksperimen ini:

```text
384 dimensi x 8 byte float64 = 3.072 byte
```

In [16]:
result_rows = []
for label, text, embedding in zip(sample_labels, sample_texts, embeddings):
    text_utf8_bytes = len(text.encode("utf-8"))
    text_token_count = len(counting_tokenizer.encode(text).ids)

    result_rows.append(
        {
            "model_name": MODEL_NAME,
            "catalog_model_size_gb": model_catalog["size_in_GB"],
            "catalog_model_size_mib": model_catalog["size_in_GB"] * 1024,
            "actual_model_size_mib": model_size_bytes / (1024**2),
            "sample": label,
            "text_characters": len(text),
            "text_token_count": text_token_count,
            "model_token_limit": token_limit,
            "token_limit_usage_pct": text_token_count / token_limit * 100,
            "text_truncated": text_token_count > token_limit,
            "embedding_data_type": str(embedding.dtype),
            "embedding_dimensions": embedding.size,
            "bytes_per_value": embedding.itemsize,
            "embedding_raw_bytes": embedding.nbytes,
            "text_plus_raw_embedding_bytes": text_utf8_bytes + embedding.nbytes,
        }
    )

result = pd.DataFrame(result_rows)

display(
    result.set_index("sample").T.style.format(
        {
            "catalog_model_size_gb": "{:,.2f}",
            "actual_model_size_mib": "{:,.2f}",
            "text_characters": "{:,.0f}",
            "text_token_count": "{:,.0f}",
            "model_token_limit": "{:,.0f}",
            "token_limit_usage_pct": "{:,.1f}%",
            "embedding_dimensions": "{:,.0f}",
            "bytes_per_value": "{:,.0f}",
            "embedding_raw_bytes": "{:,.0f}",
            "text_plus_raw_embedding_bytes": "{:,.0f}",
        }
    )
)

sample,short,long
model_name,sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2,sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
catalog_model_size_gb,0.220000,0.220000
catalog_model_size_mib,225.280000,225.280000
actual_model_size_mib,240.458208,240.458208
text_characters,27,2229
text_token_count,8,387
model_token_limit,128,128
token_limit_usage_pct,6.250000,302.343750
text_truncated,False,True
embedding_data_type,float64,float64


### **Kesimpulan**

FastEmbed mencantumkan ukuran katalog model paraphrase-multilingual-MiniLM-L12-v2 sekitar 0,22 GB. File ONNX utamanya berukuran 224,16 MiB, sedangkan keseluruhan direktori model lokal berukuran 240,46 MiB setelah menyertakan tokenizer dan file konfigurasi.

Teks pendek terdiri dari 8 token sehingga seluruh teks dapat diproses. Teks panjang terdiri dari 387 token, sedangkan batas efektif model pada konfigurasi FastEmbed ini adalah 128 token. Akibatnya, teks panjang dipotong dan hanya sekitar 33% token awal yang digunakan, sementara 259 token atau sekitar 67% tidak ikut membentuk embedding.
Walaupun panjang input berbeda, keduanya menghasilkan output dengan ukuran sama:
```go
384 dimensi × 8 byte float64 = 3.072 byte
```

Artinya, panjang teks tidak memengaruhi ukuran satu embedding. Perbedaan text_plus_raw_embedding_bytes, yaitu 3.099 byte untuk teks pendek dan 5.301 byte untuk teks panjang, hanya berasal dari ukuran teks asli.
Untuk dokumen panjang, chunking diperlukan jika seluruh bagian teks harus direpresentasikan. Setiap chunk akan menghasilkan embedding 384 dimensi baru sehingga kebutuhan penyimpanan bertambah sesuai jumlah chunk.

# Pengukuran penyimpanan embedding di OpenSearch

Note:
- Model OpenSearch perlu disiapkan terlebih dahulu dengan menjalankan
`uv run python -m index_memory_usage.opensearch_ml_model`, lalu salin model ID
yang dihasilkan ke `OPENSEARCH_MODEL_ID`.
- OpenSearch membuat embedding baru untuk sampel pendek dan panjang melalui ML
Commons Predict API. Setiap embedding kemudian dimasukkan ke indeks tersendiri.
Dokumen indeks hanya berisi field `description_embedding`. Walaupun setiap indeks hanya berisi satu dokumen, `primary_store_bytes` juga mencakup fixed overhead dari Lucene, jadi ukuran aktual ini tidak murni dari model embedding.

## 6. Hubungkan ke OpenSearch

OpenSearch harus berjalan di `OPENSEARCH_URL`. Kedua indeks benchmark bersifat
disposable dan akan dibuat ulang setiap kali bagian ini dijalankan.

In [17]:
OPENSEARCH_HOST = "localhost"
OPENSEARCH_PORT = 9200
OPENSEARCH_URL = f"http://{OPENSEARCH_HOST}:{OPENSEARCH_PORT}"
OPENSEARCH_MODEL_ID = "QEj06qABHp7iMTkBhyaM"

# Each run recreates one dedicated index for each existing sample.
OPENSEARCH_INDEX_PREFIX = "embedding-size-benchmark"

In [18]:
client = OpenSearch(
    hosts=[{"host": OPENSEARCH_HOST, "port": OPENSEARCH_PORT}],
    use_ssl=False,
    verify_certs=False,
)

cluster_info = client.info()
print(
    f"Connected to OpenSearch {cluster_info['version']['number']} at {OPENSEARCH_URL}."
)

Connected to OpenSearch 3.8.0 at http://localhost:9200.


## 7. Generate embedding dengan model OpenSearch dan mapping vector

In [19]:
if not OPENSEARCH_MODEL_ID:
    raise RuntimeError(
        "Run `uv run python -m index_memory_usage.opensearch_ml_model`, "
        "then copy the resulting model ID to OPENSEARCH_MODEL_ID."
    )


def predict_opensearch_embedding(text: str) -> list[float]:
    response = client.transport.perform_request(
        "POST",
        f"/_plugins/_ml/_predict/text_embedding/{OPENSEARCH_MODEL_ID}",
        body={
            "text_docs": [text],
            "return_number": True,
            "target_response": ["sentence_embedding"],
        },
    )

    inference_results = response.get("inference_results", [])
    if not inference_results:
        raise RuntimeError(f"Prediction returned no inference results: {response}")

    for output in inference_results[0].get("output", []):
        if output.get("name") == "sentence_embedding":
            embedding = output.get("data")
            if isinstance(embedding, list):
                return embedding

    raise RuntimeError(f"Prediction returned no sentence embedding: {response}")


opensearch_embeddings = [predict_opensearch_embedding(text) for text in sample_texts]
vector_dimension = len(opensearch_embeddings[0])

if vector_dimension != model_catalog["dim"]:
    raise RuntimeError(
        f"OpenSearch produced {vector_dimension} dimensions; "
        f"expected {model_catalog['dim']} for {MODEL_NAME!r}."
    )


def embedding_index_mapping() -> dict[str, Any]:
    return {
        "settings": {
            "index": {
                "knn": True,
                "number_of_shards": 1,
                "number_of_replicas": 0,
            }
        },
        "mappings": {
            "properties": {
                "description_embedding": {
                    "type": "knn_vector",
                    "dimension": vector_dimension,
                    "data_type": "float",
                    "method": {
                        "name": "hnsw",
                        "space_type": "cosinesimil",
                        "engine": "lucene",
                    },
                }
            }
        },
    }

## 8. Indexing dan pengukuran primary store

Setiap indeks berisi tepat satu dokumen dan satu embedding. Force merge dan
flush dilakukan sebelum ukuran primary store dibaca.

Mapping OpenSearch menggunakan `data_type: float`, yaitu `float32` dengan
ukuran 4 byte per nilai. Raw embedding dihitung dengan:

```python
embedding_raw_bytes = len(embedding) * 4
```

Pada model ini hasilnya adalah `384 x 4 byte = 1.536 byte`.

In [20]:
opensearch_rows = []

for sample, text, embedding in zip(
    sample_labels,
    sample_texts,
    opensearch_embeddings,
):
    index_name = f"{OPENSEARCH_INDEX_PREFIX}-{sample}"

    if client.indices.exists(index=index_name):
        client.indices.delete(index=index_name)

    client.indices.create(
        index=index_name,
        body=embedding_index_mapping(),
    )

    client.index(
        index=index_name,
        id=sample,
        body={"description_embedding": embedding},
        params={"refresh": "true"},
    )

    client.indices.forcemerge(
        index=index_name,
        params={"max_num_segments": 1},
    )
    client.indices.flush(index=index_name)

    index_stats = client.indices.stats(index=index_name, metric="store")
    primary_store_bytes = index_stats["_all"]["primaries"]["store"]["size_in_bytes"]

    opensearch_rows.append(
        {
            "sample": sample,
            "index_name": index_name,
            "text_characters": len(text),
            "text_utf8_bytes": len(text.encode("utf-8")),
            "embedding_dimensions": len(embedding),
            "embedding_data_type": "float32",
            "embedding_raw_bytes": len(embedding) * 4,
            "primary_store_bytes": primary_store_bytes,
        }
    )

opensearch_result = pd.DataFrame(opensearch_rows)
opensearch_result["primary_store_mib"] = opensearch_result["primary_store_bytes"] / (
    1024**2
)
opensearch_result["index_overhead_bytes"] = (
    opensearch_result["primary_store_bytes"] - opensearch_result["embedding_raw_bytes"]
)

short_index_bytes = int(
    opensearch_result.loc[
        opensearch_result["sample"] == "short",
        "primary_store_bytes",
    ].iloc[0]
)

display(
    opensearch_result.set_index("sample").T.style.format(
        {
            "text_characters": "{:,.0f}",
            "text_utf8_bytes": "{:,.0f}",
            "embedding_dimensions": "{:,.0f}",
            "embedding_raw_bytes": "{:,.0f}",
            "primary_store_bytes": "{:,.0f}",
            "primary_store_mib": "{:,.3f}",
            "index_overhead_bytes": "{:,.0f}",
        }
    )
)

sample,short,long
index_name,embedding-size-benchmark-short,embedding-size-benchmark-long
text_characters,27,2229
text_utf8_bytes,27,2229
embedding_dimensions,384,384
embedding_data_type,float32,float32
embedding_raw_bytes,1536,1536
primary_store_bytes,6492,6492
primary_store_mib,0.006191,0.006191
index_overhead_bytes,4956,4956


### **Kesimpulan**

- Teks pendek berukuran 27 byte, sedangkan teks panjang 2.229 byte.
- Keduanya tetap menghasilkan embedding berukuran sama:
  - 384 dimensi
  - float32
  - 1.536 byte per embedding
- Ukuran indeks OpenSearch yang hanya menyimpan satu dokumen berisi embedding juga identik, yaitu 6.492 byte.

Dari ukuran indeks 6.492 byte:
- Raw vector => 1.536 byte  (23,7%)
- Overhead indeks => 4.956 byte  (76,3%)
- Total primary store => 6.492 byte

index_overhead_bytes mencakup salinan vector di _source, struktur vector/Lucene, dan fixed overhead indeks satu dokumen dari HNSW. Oleh karena itu, ukurannya meningkat dari raw embedding, tapi baik teks panjang dan pendek ukurannya masih sama.